In [ ]:
import numpy as np
from numba import cuda

In [ ]:
!uv pip install -q --system numba-cuda==0.4.0

In [ ]:
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
config.CUDA_LOW_OCCUPANCY_WARNINGS = 0

$
\begin{bmatrix}
 a_{11} &  \dots  & a_{1n} \\
 \vdots &  \ddots & \vdots \\
 a_{m1} &  \dots  & a_{mn}
\end{bmatrix}
+
\begin{bmatrix}
 b_{11} & \dots  & b_{1n} \\
 \vdots & \ddots & \vdots \\
 b_{m1} & \dots  & b_{mn}
\end{bmatrix}
=\begin{bmatrix}
 a_{11} + b_{11} & \dots  & a_{1n} + b_{1n} \\
     \vdots      & \ddots &     \vdots      \\
 a_{m1} + b_{m1} & \dots  & a_{mn} + b_{mn}
\end{bmatrix}
$

In [ ]:
def add_mat(A,B):
    dim1,dim2 = A.shape
    C = np.empty_like(A)
    for i in range(dim1):
        for j in range(dim2):
            C[i,j] = A[i,j] + B[i,j]
    return C

#Complete the kernel

In [ ]:
@cuda.jit
def k_add_mat(a, b, c):
    i = cuda.threadIdx.x
    j = cuda.threadIdx.y
    c[i,j] = a[i,j] + b[i,j]

#Matrices initialization on host

In [ ]:
h_a = np.random.randn(5,5)
h_b = np.random.randn(5,5)
h_a = h_a.astype(np.float32)
h_b = h_b.astype(np.float32)

# Complete memory allocation and copy to device

In [ ]:
#copy A to g_A on device
g_A = cuda.to_device(h_a)
#copy B to g_B on device
g_B = cuda.to_device(h_b)
# Allocate memory for g_C
nrow, ncol = h_a.shape
g_C = cuda.device_array((nrow,ncol))

# Call the kernel with well-sized block

In [ ]:
k_add_mat[1,(nrow,ncol)](g_A, g_B, g_C)

# Copy results to the host

In [ ]:
h_c = g_C.copy_to_host()

# Check results

In [ ]:
np.array_equal(h_a + h_b, h_c)

# Modify the kernel `k_add_huge_mat` to manage this matrices

In [ ]:
@cuda.jit
def k_add_huge_mat(a, b, c, width):
    i = cuda.threadIdx.x
    j = cuda.threadIdx.y

    if i < a.shape[0] and j < a.shape[1]:
        c[i, j] = a[i, j] + b[i, j]

In [ ]:
h_HA = np.random.randn(1024,1024)
h_HB = np.random.randn(1024,1024)
h_HA = h_HA.astype(np.float32)
h_HB = h_HB.astype(np.float32)

In [ ]:
g_HA = cuda.to_device(h_HA)
g_HB = cuda.to_device(h_HB)
dim1,dim2=h_HA.shape
g_HC = cuda.device_array((dim1,dim2))

In [ ]:
dim1,dim2=h_HA.shape
# chaque thread prend en charge width operations dans le cas où l'opération
#   tombe juste width = nombre d'éléments par matrice / nombre de threads
width=np.int32((dim1 * dim2) / (32 * 32))

In [ ]:
k_add_huge_mat[1,(np.sqrt(width),np.sqrt(width))](g_HA,g_HB,g_HC,width)

In [ ]:
h_HC = g_HC.copy_to_host()

In [ ]:
np.array_equal(h_a + h_b, h_c)

In [ ]:
import time
tic = time.time()
add_mat(g_HA,g_HB)
toc = time.time()
print(f"Temps d'execution = {toc-tic}s")

In [ ]:
tic = time.time()
g_HA = cuda.to_device(h_HA)
g_HB = cuda.to_device(h_HB)
dim1,dim2=h_HA.shape
g_HC = cuda.device_array((dim1,dim2))
k_add_huge_mat[1,(np.sqrt(width),np.sqrt(width))](g_HA,g_HB,g_HC,width)
h_HC = g_HC.copy_to_host()
toc = time.time()
print(f"Temps d'execution = {toc-tic}s")

In [ ]:
tic = time.time()
h_HA+h_HB
toc = time.time()
print(f"Temps d'execution = {toc-tic}s")